In [ ]:
import pickle
import pandas as pd
from pathlib import Path
from typing import Final
import sys
import os

CURRENT_DIR: Final[Path] = Path(os.getcwd())
PROJECT_DIR: Final[Path] = CURRENT_DIR.parent
sys.path.append(str(PROJECT_DIR))


In [ ]:
from log_parser import LogParser
from config import LOOP_KEYWORDS, MAIN_KEYWORD


In [ ]:
pd.set_option("display.float_format", lambda x: "%.3f" % x)

In [ ]:
def q25(x: pd.Series) -> float:
    return x.quantile(0.25)


def q50(x: pd.Series) -> float:
    return x.quantile(0.50)


def q75(x: pd.Series) -> float:
    return x.quantile(0.75)


def q90(x: pd.Series) -> float:
    return x.quantile(0.90)


def q95(x: pd.Series) -> float:
    return x.quantile(0.95)


def q99(x: pd.Series) -> float:
    return x.quantile(0.99)

# Generated Performance Statistics Data

In [ ]:
log_path = "/home/gpal/workspace/spatiotemporal_planner/hil_logs/eka-rt-log_r"

stat_result_name = "main_stat_result.pkl"
database_path = CURRENT_DIR / stat_result_name


ps = LogParser(log_path)
ps.run(database_path, MAIN_KEYWORD, LOOP_KEYWORDS)


# Load Performance Statistics Data

In [ ]:
with open(database_path, "rb") as f:
    database = pd.DataFrame(pickle.load(f))

# Analyze the Performance Statistics Data

In [ ]:
database.info()

In [ ]:
database.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.9999])

In [ ]:
import pandas as pd
import altair as alt
import pickle
from pathlib import Path

# --- 步骤 1: 加载性能统计数据 ---
CURRENT_DIR = Path.cwd() 
stat_result_name = "main_stat_result.pkl"
database_path = CURRENT_DIR / stat_result_name

database = pd.DataFrame()
try:
    with open(database_path, "rb") as f:
        loaded_data = pickle.load(f)
        if isinstance(loaded_data, dict):
            database = pd.DataFrame(loaded_data)
        elif isinstance(loaded_data, pd.DataFrame):
            database = loaded_data
        else:
            print(f"错误：加载的文件 '{stat_result_name}' 不是一个有效的数据格式。")
except FileNotFoundError:
    print(f"错误：找不到统计结果文件 '{stat_result_name}'。")
    print("请确保您已经成功运行了本 Notebook 前面的单元格来生成该文件。")
except Exception as e:
    print(f"加载文件时发生未知错误: {e}")

if not database.empty:
    # --- 步骤 2: 准备数据 ---
    db_with_cycle = database.reset_index().rename(columns={'index': '周期 (Cycle)'})
    df_long = db_with_cycle.melt(
        id_vars=['周期 (Cycle)'], 
        var_name='子模块 (Submodule)', 
        value_name='耗时 (ms)'
    )

    # --- 步骤 3: 创建支持两种交互的图表 ---
    
    # 交互功能1：用于在X轴上平移和缩放
    zoom_selection = alt.selection_interval(bind='scales', encodings=['x'])
    
    # 交互功能2：用于点击图例来筛选模块。
    legend_selection = alt.selection_multi(
        fields=['子模块 (Submodule)'], 
        bind='legend'
    )

    chart = alt.Chart(df_long).mark_rule().encode(
        x=alt.X('周期 (Cycle):Q', title='统计周期 (Cycle)'),
        y=alt.Y('耗时 (ms):Q', title='耗时 (ms)', scale=alt.Scale(zero=False)),
        color=alt.Color('子模块 (Submodule):N', title='子模块'),
        opacity=alt.condition(legend_selection, alt.value(1.0), alt.value(0.1)),
        tooltip=[
            '周期 (Cycle)',
            '子模块 (Submodule)',
            alt.Tooltip('耗时 (ms)', format='.3f')
        ]
    ).properties(
        title='各子模块逐周期耗时分布 (可平移缩放、可点击筛选)',
        width=800,
        height=500
    ).add_selection(
        zoom_selection,
        legend_selection
    )

    # --- 步骤 4: 将图表保存为独立的 HTML 文件 ---
    html_filename = 'interactive_chart_with_toggle.html'
    try:
        chart.save(html_filename)
        print(f"🎉 耗时统计网页图表已成功生成！")
        print(f"请在文件浏览器中找到 '{html_filename}'，然后直接双击打开它。")
    except Exception as e:
        print(f"保存HTML文件时出错: {e}")